# House Price Prediction – Machine Learning Regression

This notebook implements a complete ML pipeline to predict house prices based on features such as house size, number of bedrooms, number of bathrooms, lot size, garage size, neighborhood quality, and the year the house was built.

**GitHub Repository:** https://github.com/anushplayer/housepriceprediction

## Step 1 – Import Required Libraries

In [ ]:
# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & modelling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 20)
sns.set_theme(style='whitegrid')
print('All libraries imported successfully.')

## Step 2 – Load the Dataset

In [ ]:
df = pd.read_csv('house_price_dataset.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## Step 3 – Check Dataset

In [ ]:
print('=== Shape ===')
print(df.shape)

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Basic Statistics ===')
df.describe(include='all')

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicate Rows ===')
print(f'Number of duplicate rows: {df.duplicated().sum()}')

## Step 4 – Handle Missing Values

In [ ]:
# Numerical columns – fill with median (robust to outliers)
num_cols_with_na = df.select_dtypes(include='number').columns[df.select_dtypes(include='number').isnull().any()]
for col in num_cols_with_na:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f'  Filled "{col}" missing values with median = {median_val:.2f}')

print('\nMissing values after imputation:')
print(df.isnull().sum())

## Step 5 – Remove Duplicate Records

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
after = len(df)
print(f'Removed {before - after} duplicate row(s). Dataset now has {after} rows.')

## Step 6 – Univariate Analysis

In [ ]:
numerical_cols = ['house_size', 'bedrooms', 'bathrooms', 'lot_size', 'garage_size', 'year_built', 'price']

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(f'Distribution of {col}', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Categorical column
axes[7].bar(
    df['neighborhood_quality'].value_counts().index,
    df['neighborhood_quality'].value_counts().values,
    color='coral', edgecolor='white'
)
axes[7].set_title('Neighborhood Quality Count', fontsize=12)
axes[7].set_xlabel('Neighborhood Quality')
axes[7].set_ylabel('Count')

axes[8].axis('off')  # empty subplot
plt.tight_layout()
plt.suptitle('Univariate Analysis', fontsize=16, y=1.01)
plt.savefig('univariate_analysis.png', bbox_inches='tight', dpi=100)
plt.show()

In [ ]:
# Box plots for spread / outlier check
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    axes[i].set_title(f'Boxplot: {col}', fontsize=11)
    axes[i].set_ylabel(col)

axes[7].axis('off')
plt.tight_layout()
plt.suptitle('Box Plots – Univariate', fontsize=15, y=1.01)
plt.savefig('boxplots_univariate.png', bbox_inches='tight', dpi=100)
plt.show()

## Step 7 – Bivariate Analysis

In [ ]:
feature_cols = ['house_size', 'bedrooms', 'bathrooms', 'lot_size', 'garage_size', 'year_built']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].scatter(df[col], df['price'], alpha=0.4, color='teal', s=15)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('price')
    axes[i].set_title(f'price vs {col}')

plt.tight_layout()
plt.suptitle('Bivariate Analysis – Feature vs Price', fontsize=15, y=1.01)
plt.savefig('bivariate_analysis.png', bbox_inches='tight', dpi=100)
plt.show()

In [ ]:
# Boxplot: price by neighborhood_quality
order = ['Low', 'Medium', 'High', 'Premium']
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='neighborhood_quality', y='price', order=order, palette='Set2')
plt.title('House Price by Neighborhood Quality')
plt.savefig('price_by_neighborhood.png', bbox_inches='tight', dpi=100)
plt.show()

## Step 8 – Correlation Heatmap

In [ ]:
plt.figure(figsize=(10, 7))
corr_matrix = df[numerical_cols].corr()
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, square=True, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=100)
plt.show()

## Step 9 – Detect and Handle Outliers

In [ ]:
def detect_outliers_iqr(data, col):
    """Return boolean mask of outlier rows using the IQR method."""
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (data[col] < lower) | (data[col] > upper)

print('Outlier counts per numerical column (IQR method):')
for col in numerical_cols:
    n_out = detect_outliers_iqr(df, col).sum()
    print(f'  {col}: {n_out} outliers')

In [ ]:
# Cap outliers using the IQR fence (Winsorisation) for numerical features
before_shape = df.shape
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print(f'Outliers capped. Dataset shape unchanged: {df.shape} (was {before_shape})')

## Step 10 – Apply Encoding

In [ ]:
print('Categorical columns:', df.select_dtypes(include='object').columns.tolist())

# Ordinal encoding for neighborhood_quality (preserves order: Low < Medium < High < Premium)
ordinal_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Premium': 3}
df['neighborhood_quality_enc'] = df['neighborhood_quality'].map(ordinal_map)
print('\nEncoded neighborhood_quality:')
print(df[['neighborhood_quality', 'neighborhood_quality_enc']].value_counts().sort_index())

## Step 11 – Apply Log Transformation

In [ ]:
# Apply log1p to right-skewed features to reduce skewness
skew_before = df[['price', 'house_size', 'lot_size']].skew()
print('Skewness BEFORE log transformation:')
print(skew_before)

for col in ['price', 'house_size', 'lot_size']:
    df[f'log_{col}'] = np.log1p(df[col])

skew_after = df[['log_price', 'log_house_size', 'log_lot_size']].skew()
print('\nSkewness AFTER log transformation:')
print(skew_after)

# Visualise the effect on price
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['price'], bins=30, color='salmon', edgecolor='white')
axes[0].set_title('Price – Original')
axes[1].hist(df['log_price'], bins=30, color='seagreen', edgecolor='white')
axes[1].set_title('Price – Log Transformed')
plt.suptitle('Log Transformation Effect on Price', fontsize=13)
plt.tight_layout()
plt.savefig('log_transformation.png', bbox_inches='tight', dpi=100)
plt.show()

## Step 12 – Separate Features and Target Variable

In [ ]:
# Use log-transformed features where applicable; encoded categorical
feature_columns = [
    'log_house_size', 'bedrooms', 'bathrooms',
    'log_lot_size', 'garage_size',
    'neighborhood_quality_enc', 'year_built'
]

X = df[feature_columns]
y = df['log_price']   # predict log-price; convert back with np.expm1 after evaluation

print('Feature matrix shape:', X.shape)
print('Target vector shape :', y.shape)
X.head()

## Step 13 – Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training set  : {X_train.shape[0]} samples')
print(f'Test set      : {X_test.shape[0]} samples')

## Step 14 – Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Features scaled with StandardScaler.')
print('X_train_scaled mean (approx):', X_train_scaled.mean(axis=0).round(4))
print('X_train_scaled std  (approx):', X_train_scaled.std(axis=0).round(4))

## Step 15 – Train Regression Models

### 15a – Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
print('Linear Regression training complete.')

# Coefficients
coef_df = pd.DataFrame({'Feature': feature_columns, 'Coefficient': lr_model.coef_})
coef_df = coef_df.sort_values('Coefficient', ascending=False)
plt.figure(figsize=(8, 5))
sns.barplot(data=coef_df, x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Linear Regression – Feature Coefficients')
plt.tight_layout()
plt.savefig('lr_coefficients.png', bbox_inches='tight', dpi=100)
plt.show()

### 15b – K-Nearest Neighbours (KNN) Regression

In [ ]:
# Tune k using a simple loop on validation set
best_k, best_r2 = 5, -np.inf
for k in range(3, 21):
    knn = KNeighborsRegressor(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    score = r2_score(y_test, knn.predict(X_test_scaled))
    if score > best_r2:
        best_k, best_r2 = k, score

print(f'Best k = {best_k} (R² = {best_r2:.4f})')

knn_model = KNeighborsRegressor(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)
print('KNN Regression training complete.')

## Step 16 – Evaluate Models

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error (working in original price space)."""
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(y_pred)
    return np.mean(np.abs((y_true_orig - y_pred_orig) / y_true_orig)) * 100

def adjusted_r2(r2, n, p):
    """Adjusted R² given R², number of samples n, and features p."""
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def evaluate(name, y_true, y_pred, n_features):
    n = len(y_true)
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape_val = mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    adj_r2 = adjusted_r2(r2, n, n_features)
    return {
        'Model': name,
        'MAE':  round(mae, 4),
        'MSE':  round(mse, 4),
        'RMSE': round(rmse, 4),
        'MAPE (%)': round(mape_val, 2),
        'R² Score': round(r2, 4),
        'Adjusted R²': round(adj_r2, 4),
    }

n_feat = len(feature_columns)
results = pd.DataFrame([
    evaluate('Linear Regression', y_test, y_pred_lr, n_feat),
    evaluate('KNN Regression',    y_test, y_pred_knn, n_feat),
])

results.set_index('Model', inplace=True)
print('=== Model Evaluation Results ===')
results

In [ ]:
# Visualise Actual vs Predicted for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name, y_pred in zip(axes,
                             ['Linear Regression', 'KNN Regression'],
                             [y_pred_lr, y_pred_knn]):
    ax.scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=15)
    mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
    ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual log(price)')
    ax.set_ylabel('Predicted log(price)')
    ax.set_title(f'{name} – Actual vs Predicted')
    ax.legend()

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', bbox_inches='tight', dpi=100)
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, name, y_pred in zip(axes,
                             ['Linear Regression', 'KNN Regression'],
                             [y_pred_lr, y_pred_knn]):
    residuals = y_test.values - y_pred
    ax.scatter(y_pred, residuals, alpha=0.4, color='darkorange', s=15)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted log(price)')
    ax.set_ylabel('Residuals')
    ax.set_title(f'{name} – Residual Plot')

plt.tight_layout()
plt.savefig('residual_plots.png', bbox_inches='tight', dpi=100)
plt.show()

In [ ]:
# Metric comparison bar chart
metrics_to_plot = ['MAE', 'RMSE', 'MAPE (%)']
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, results.loc['Linear Regression', metrics_to_plot], width,
               label='Linear Regression', color='steelblue')
bars2 = ax.bar(x + width/2, results.loc['KNN Regression',    metrics_to_plot], width,
               label='KNN Regression', color='coral')

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_title('Model Comparison – Error Metrics')
ax.legend()
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=100)
plt.show()

print('\nFinal Results Summary:')
print(results.to_string())

## Conclusion

| Metric | Linear Regression | KNN Regression |
|--------|:-----------------:|:--------------:|
| Lower error metrics are better (MAE, MSE, RMSE, MAPE) | ✓ compared above | ✓ compared above |
| Higher R² / Adjusted R² is better | ✓ compared above | ✓ compared above |

**Key observations:**
- **Linear Regression** benefits from its simplicity and interpretability; the log-transformed features improve linearity.
- **KNN Regression** is a non-parametric model that can capture non-linear relationships; performance depends on the chosen *k* and feature scaling.
- Both models were evaluated using MAE, MSE, RMSE, MAPE, R² Score, and Adjusted R² to give a comprehensive picture of performance.

**GitHub Repository:** https://github.com/anushplayer/housepriceprediction